In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 7.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import RidgeCV, ElasticNetCV
import re
import warnings
warnings.filterwarnings('ignore')

# -----------------------------------------------------------------
# 1. Load and Clean Data
# -----------------------------------------------------------------
print("Step 1: Loading and Cleaning Data...")

try:
    train_df = pd.read_csv('datatrain.csv')
    test_df = pd.read_csv('datatest.csv')
    submission_dates = test_df['date'].copy()
except FileNotFoundError:
    print("Make sure 'datatrain.csv' and 'datatest.csv' are in the same directory.")
    exit()

train_ids = train_df['obs_id']
test_ids = test_df['obs_id']
train_y = train_df['daily_ktCO2']

train_df = train_df.drop(columns=['daily_ktCO2', 'obs_id'])
test_df = test_df.drop(columns=['obs_id'])

combined_df = pd.concat([train_df, test_df], axis=0, ignore_index=True)

text_cols = combined_df.select_dtypes(include=['object']).columns
for col in text_cols:
    if col != 'date':
        combined_df[col] = combined_df[col].astype(str).str.lower().str.strip()

numeric_cols_to_convert = ['pm25', 'pm10', 'o3', 'no2', 'so2', 'env_index', 'temp', 'tempmax', 'tempmin', 'temp_calib', 'feelslikemax', 'feelslikemin', 'feelslike', 'dew', 'humidity', 'precip', 'precipprob', 'precipcover', 'windgust', 'windspeed', 'windspeedmax', 'windspeedmean', 'windspeedmin', 'winddir', 'sealevelpressure', 'cloudcover', 'visibility', 'solarradiation', 'solarenergy', 'uvindex', 'moonphase', 'avg_ridership_monthly', 'avg_ridership_workday_monthly', 'TCI']
traffic_pattern_cols = [col for col in combined_df.columns if any(pattern in col for pattern in ['7-9', '9-17', '17-19'])]
numeric_cols_to_convert.extend(traffic_pattern_cols)
numeric_cols_to_convert = list(pd.Series(numeric_cols_to_convert).drop_duplicates())

for col in numeric_cols_to_convert:
    if col in combined_df.columns:
        combined_df[col] = pd.to_numeric(combined_df[col], errors='coerce')

bool_cols = ['rain', 'weekend', 'holiday']
for col in bool_cols:
    if col in combined_df.columns:
        map_dict = {'true': True, 'yes': True, '1': True, 'false': False, 'no': False, '0': False}
        combined_df[col] = combined_df[col].astype(str).str.strip().str.lower().map(map_dict)
        combined_df[col] = combined_df[col].fillna(False).astype(int)

# -----------------------------------------------------------------
# 2. ADVANCED Feature Engineering
# -----------------------------------------------------------------
print("\nStep 2: Advanced Feature Engineering...")

combined_df['date'] = pd.to_datetime(combined_df['date'])
combined_df['month'] = combined_df['date'].dt.month
combined_df['day_of_week'] = combined_df['date'].dt.dayofweek
combined_df['day_of_year'] = combined_df['date'].dt.dayofyear
combined_df['week_of_year'] = combined_df['date'].dt.isocalendar().week.astype(int)
combined_df['year'] = combined_df['date'].dt.year
combined_df['quarter'] = combined_df['date'].dt.quarter
combined_df['is_weekend'] = (combined_df['day_of_week'] >= 5).astype(int)
combined_df['is_month_start'] = combined_df['date'].dt.is_month_start.astype(int)
combined_df['is_month_end'] = combined_df['date'].dt.is_month_end.astype(int)
combined_df['days_since_start'] = (combined_df['date'] - combined_df['date'].min()).dt.days

combined_df['is_quarter_start'] = combined_df['date'].dt.is_quarter_start.astype(int)
combined_df['is_quarter_end'] = combined_df['date'].dt.is_quarter_end.astype(int)
combined_df['days_in_month'] = combined_df['date'].dt.days_in_month
combined_df['week_of_month'] = (combined_df['date'].dt.day - 1) // 7 + 1

combined_df = combined_df.drop('date', axis=1)

def add_cyclical_features(df, col, period):
    if col in df.columns:
        df[f'{col}_sin'] = np.sin(2 * np.pi * df[col] / period)
        df[f'{col}_cos'] = np.cos(2 * np.pi * df[col] / period)
        df[f'{col}_sin2'] = np.sin(4 * np.pi * df[col] / period)
        df[f'{col}_cos2'] = np.cos(4 * np.pi * df[col] / period)
    return df

if 'winddir' in combined_df.columns:
    combined_df = add_cyclical_features(combined_df, 'winddir', 360)
    combined_df = combined_df.drop('winddir', axis=1)

combined_df = add_cyclical_features(combined_df, 'month', 12)
combined_df = add_cyclical_features(combined_df, 'day_of_week', 7)
combined_df = add_cyclical_features(combined_df, 'day_of_year', 365)
combined_df = add_cyclical_features(combined_df, 'week_of_year', 52)

cyclical_to_drop = ['month', 'day_of_week', 'day_of_year', 'week_of_year']
combined_df = combined_df.drop(columns=[col for col in cyclical_to_drop if col in combined_df.columns])

traffic_cols = [col for col in combined_df.columns if any(time in col for time in ['7-9', '9-17', '17-19'])]
vehicle_types = ['Car', 'Van', 'Bus', 'Minibus', 'Truck', '3Cycle']
for vehicle in vehicle_types:
    vehicle_cols = [col for col in traffic_cols if vehicle in col]
    if len(vehicle_cols) > 0:
        combined_df[f'{vehicle}_total'] = combined_df[vehicle_cols].sum(axis=1)
        combined_df[f'{vehicle}_peak_ratio'] = (combined_df.get(f'{vehicle}_7-9', 0) + combined_df.get(f'{vehicle}_17-19', 0)) / (combined_df[f'{vehicle}_total'] + 1e-6)

morning_cols = [col for col in traffic_cols if '7-9' in col]
day_cols = [col for col in traffic_cols if '9-17' in col]
evening_cols = [col for col in traffic_cols if '17-19' in col]

if morning_cols:
    combined_df['traffic_morning'] = combined_df[morning_cols].sum(axis=1)
else:
    combined_df['traffic_morning'] = 0
if day_cols:
    combined_df['traffic_day'] = combined_df[day_cols].sum(axis=1)
else:
    combined_df['traffic_day'] = 0
if evening_cols:
    combined_df['traffic_evening'] = combined_df[evening_cols].sum(axis=1)
else:
    combined_df['traffic_evening'] = 0

combined_df['traffic_total'] = (combined_df['traffic_morning'] + combined_df['traffic_day'] + combined_df['traffic_evening'])
combined_df['rush_hour_ratio'] = ((combined_df['traffic_morning'] + combined_df['traffic_evening']) / (combined_df['traffic_total'] + 1e-6))

weather_cols = ['temp', 'tempmax', 'tempmin', 'humidity', 'windspeed', 'sealevelpressure']
pollutant_cols = ['pm25', 'pm10', 'o3', 'no2', 'so2']
if all(col in combined_df.columns for col in ['temp', 'tempmax', 'tempmin']):
    combined_df['temp_range'] = combined_df['tempmax'] - combined_df['tempmin']
    combined_df['temp_variance'] = combined_df['temp_range'] / (combined_df['temp'].replace(0, 1e-6) + 1e-6)
if 'temp' in combined_df.columns and 'humidity' in combined_df.columns:
    combined_df['heat_index'] = combined_df['temp'] * (1 + combined_df['humidity'] / 100)
    combined_df['comfort_index'] = combined_df['temp'] * (1 - abs(combined_df['humidity'] - 50) / 100)

wind_features = ['windspeed', 'windgust', 'windspeedmax', 'windspeedmin']
for feat in wind_features:
    if feat in combined_df.columns and 'temp' in combined_df.columns:
        combined_df[f'{feat}_temp_interaction'] = combined_df[feat] * combined_df['temp']

if 'sealevelpressure' in combined_df.columns:
    combined_df['pressure_anomaly'] = combined_df['sealevelpressure'] - combined_df['sealevelpressure'].mean()

if len([col for col in pollutant_cols if col in combined_df.columns]) > 1:
    available_pollutants = [col for col in pollutant_cols if col in combined_df.columns]
    combined_df['pollutant_sum'] = combined_df[available_pollutants].sum(axis=1)
    combined_df['pollutant_mean'] = combined_df[available_pollutants].mean(axis=1)
    if 'pm25' in combined_df.columns and 'pm10' in combined_df.columns:
        combined_df['fine_coarse_ratio'] = combined_df['pm25'] / (combined_df['pm10'] + 1e-6)
    if 'no2' in combined_df.columns and 'o3' in combined_df.columns:
        combined_df['no2_o3_ratio'] = combined_df['no2'] / (combined_df['o3'] + 1e-6)

solar_features = ['solarradiation', 'solarenergy', 'uvindex']
for feat in solar_features:
    if feat in combined_df.columns and 'cloudcover' in combined_df.columns:
        combined_df[f'{feat}_cloud_interaction'] = combined_df[feat] * (100 - combined_df['cloudcover']) / 100

ridership_cols = ['avg_ridership_monthly', 'avg_ridership_workday_monthly']
for col in ridership_cols:
    if col in combined_df.columns:
        combined_df[f'{col}_per_traffic'] = combined_df[col] / (combined_df.get('traffic_total', 1) + 1e-6)

# -----------------------------------------------------------------
# 3. SMART Missing Value Handling
# -----------------------------------------------------------------
print("\nStep 3: Advanced Missing Value Handling...")

high_missing_cols = [col for col in combined_df.columns if combined_df[col].isnull().sum() / len(combined_df) > 0.7]
if high_missing_cols:
    combined_df = combined_df.drop(columns=high_missing_cols)
    print(f"Dropped {len(high_missing_cols)} columns with >70% missing data")

numerical_cols = combined_df.select_dtypes(include=[np.number]).columns.tolist()
if 'month_sin' in combined_df.columns:
    weather_seasonal_cols = [col for col in numerical_cols if any(weather in col.lower() for weather in ['temp', 'humid', 'wind', 'pressure', 'solar', 'precip', 'uvindex', 'dew'])]
    for col in weather_seasonal_cols:
        if combined_df[col].isnull().any():
            bins = np.linspace(combined_df['month_sin'].min(), combined_df['month_sin'].max(), 5)
            if combined_df['month_sin'].nunique() > 1:
                try:
                    for _, season_group_df in combined_df.groupby(pd.cut(combined_df['month_sin'], bins=bins, include_lowest=True)):
                        mask = season_group_df.index
                        if combined_df.loc[mask, col].isnull().any():
                            fill_val = combined_df.loc[mask, col].median()
                            if pd.isna(fill_val):
                                fill_val = combined_df[col].median()
                            combined_df.loc[mask, col] = combined_df.loc[mask, col].fillna(fill_val)
                except Exception:
                    fill_val = combined_df[col].median()
                    combined_df[col] = combined_df[col].fillna(fill_val)
            else:
                fill_val = combined_df[col].median()
                combined_df[col] = combined_df[col].fillna(fill_val)
else:
    print("Warning: 'month_sin' not found for seasonal imputation. Skipping seasonal imputation for weather features.")

numerical_cols = combined_df.select_dtypes(include=[np.number]).columns.tolist()
for col in numerical_cols:
    if combined_df[col].isnull().any():
        fill_val = combined_df[col].median()
        combined_df[col] = combined_df[col].fillna(fill_val)

categorical_cols = combined_df.select_dtypes(include=['object', 'bool']).columns.tolist()
for col in categorical_cols:
    if combined_df[col].isnull().any():
        mode_vals = combined_df[col].mode()
        fill_val = mode_vals[0] if len(mode_vals) > 0 else 'unknown'
        combined_df[col] = combined_df[col].fillna(fill_val)

# -----------------------------------------------------------------
# 4. ADVANCED Feature Selection and Preprocessing
# -----------------------------------------------------------------
print("\nStep 4: Advanced Feature Selection...")

from sklearn.feature_selection import VarianceThreshold
numerical_cols = combined_df.select_dtypes(include=[np.number]).columns
if len(numerical_cols) > 0:
    selector = VarianceThreshold(threshold=0.005)
    selector.fit(combined_df[numerical_cols])
    low_var_mask = selector.get_support()
    low_var_features = numerical_cols[~low_var_mask]
    if len(low_var_features) > 0:
        combined_df = combined_df.drop(columns=low_var_features)

numerical_cols = combined_df.select_dtypes(include=[np.number]).columns
if len(numerical_cols) > 1:
    corr_matrix = combined_df[numerical_cols].corr().abs()
    features_to_drop = set()
    for i in range(len(numerical_cols)):
        for j in range(i + 1, len(numerical_cols)):
            col1 = numerical_cols[i]
            col2 = numerical_cols[j]
            if corr_matrix.iloc[i, j] > 0.85:
                if combined_df[col1].var() < combined_df[col2].var():
                    features_to_drop.add(col1)
                else:
                    features_to_drop.add(col2)
    if features_to_drop:
        combined_df = combined_df.drop(columns=list(features_to_drop))

categorical_cols = combined_df.select_dtypes(include=['object', 'bool']).columns.tolist()
if categorical_cols:
    for col in categorical_cols:
        if combined_df[col].nunique() > 10:
            freq_map = combined_df[col].value_counts(normalize=True).to_dict()
            combined_df[f'{col}_freq'] = combined_df[col].map(freq_map)
            combined_df = combined_df.drop(columns=[col])
        else:
            if combined_df[col].isnull().any():
                combined_df[col] = combined_df[col].fillna('Missing')
            combined_df = pd.get_dummies(combined_df, columns=[col], prefix=col, drop_first=True)

combined_df.columns = [re.sub(r'[^A-Za-z0-9_]', '_', col) for col in combined_df.columns]
combined_df.columns = [re.sub(r'_+', '_', col).strip('_') for col in combined_df.columns]

from sklearn.ensemble import IsolationForest
numerical_cols = combined_df.select_dtypes(include=[np.number]).columns
if len(numerical_cols) > 1 and len(combined_df) > 100:
    iso_forest = IsolationForest(contamination=0.05, random_state=42, n_jobs=-1)
    outliers = iso_forest.fit_predict(combined_df[numerical_cols])
    for col in numerical_cols:
        lower_bound = combined_df[col].quantile(0.01)
        upper_bound = combined_df[col].quantile(0.99)
        combined_df[col] = np.clip(combined_df[col], lower_bound, upper_bound)

# Separate train/test again
train_len = len(train_y)
X_train = combined_df.iloc[:train_len].copy()
X_test = combined_df.iloc[train_len:].copy()

print(f"\nFinal Training set shape: {X_train.shape}")
print(f"Final Test set shape: {X_test.shape}")

Step 1: Loading and Cleaning Data...

Step 2: Advanced Feature Engineering...

Step 3: Advanced Missing Value Handling...
Dropped 28 columns with >70% missing data

Step 4: Advanced Feature Selection...

Final Training set shape: (1606, 61)
Final Test set shape: (402, 61)


In [ ]:
# Import tambahan yang diperlukan
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import RidgeCV
import lightgbm as lgb
import numpy as np

# -----------------------------------------------------------------
# 5. Simple Stacking with Enhanced Base Models
# -----------------------------------------------------------------
print("\n" + "="*80)
print("STACKING ENSEMBLE DENGAN PENINGKATAN OPTIMASI")
print("="*80)

# Mendefinisikan model dasar yang ditingkatkan dengan parameter yang masuk akal
base_models = {
    'lgbm': lgb.LGBMRegressor(random_state=42, n_estimators=1500, learning_rate=0.03, num_leaves=31, verbose=-1),
    'xgb': xgb.XGBRegressor(random_state=42, n_estimators=1500, learning_rate=0.03, max_depth=6, verbosity=0),
    'rf': RandomForestRegressor(random_state=42, n_estimators=300, max_depth=15, n_jobs=-1),
    'cat': cb.CatBoostRegressor(random_state=42, iterations=1500, learning_rate=0.03, depth=8, verbose=0),
    'svr': Pipeline([('scaler', StandardScaler()), ('svr', SVR(C=1.0, epsilon=0.1))]),
    'mlp': Pipeline([('scaler', StandardScaler()), ('mlp', MLPRegressor(hidden_layer_sizes=(100, 50), activation='relu', solver='adam', alpha=0.001, max_iter=500, random_state=42))])
}

# Mengatur K-Fold Cross-Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Membuat array untuk menyimpan prediksi out-of-fold dan prediksi set pengujian
oof_preds = np.zeros((X_train.shape[0], len(base_models)))
test_preds = np.zeros((X_test.shape[0], len(base_models)))
oof_rmse_scores = {}

print("--- Melatih Model Dasar dengan Parameter Manual ---")

for i, (name, model) in enumerate(base_models.items()):
    print(f"Melatih {name}...")
    
    # Melatih model pada setiap fold untuk OOF predictions
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = train_y.iloc[train_idx], train_y.iloc[val_idx]
        
        if name in ['cat']:
            model.fit(X_fold_train, y_fold_train,
                      eval_set=[(X_fold_val, y_fold_val)],
                      early_stopping_rounds=50, verbose=False)
        else:
            model.fit(X_fold_train, y_fold_train)
            
        oof_preds[val_idx, i] = model.predict(X_fold_val)

    # Melatih model pada seluruh data pelatihan dan memprediksi data uji
    model.fit(X_train, train_y)
    test_preds[:, i] = model.predict(X_test)

    # Menghitung dan mencetak OOF RMSE untuk model dasar
    oof_rmse = np.sqrt(mean_squared_error(train_y, oof_preds[:, i]))
    oof_rmse_scores[name] = oof_rmse
    print(f"  {name:<4} OOF RMSE: {oof_rmse:.4f}")

# -----------------------------------------------------------------
# 6. Membandingkan Ensemble vs. Stacking
# -----------------------------------------------------------------
print("\n--- Membandingkan Ensemble Pembobotan & Stacking ---")

# 1. Mengoptimalkan Ensemble dengan Model Linier
# Menggunakan RidgeCV untuk menemukan bobot optimal dari prediksi OOF
ridge_blender = RidgeCV(alphas=np.logspace(-5, 5, 50))
ridge_blender.fit(oof_preds, train_y)

weighted_ensemble_oof = ridge_blender.predict(oof_preds)
weighted_ensemble_test = ridge_blender.predict(test_preds)
weighted_ensemble_rmse = np.sqrt(mean_squared_error(train_y, weighted_ensemble_oof))

print(f"  Optimal Weighted Ensemble OOF RMSE: {weighted_ensemble_rmse:.4f}")

# 2. Meningkatkan Stacking Meta-Model
# Menambahkan fitur interaksi
oof_preds_ext = np.hstack([oof_preds, oof_preds[:, 0].reshape(-1,1) * oof_preds[:, 2].reshape(-1,1), oof_preds[:, 0].reshape(-1,1) * oof_preds[:, 3].reshape(-1,1)])
test_preds_ext = np.hstack([test_preds, test_preds[:, 0].reshape(-1,1) * test_preds[:, 2].reshape(-1,1), test_preds[:, 0].reshape(-1,1) * test_preds[:, 3].reshape(-1,1)])

meta_model_lgbm = lgb.LGBMRegressor(random_state=42, n_estimators=500, learning_rate=0.01)
meta_model_lgbm.fit(oof_preds_ext, train_y)
stacking_oof_preds = meta_model_lgbm.predict(oof_preds_ext)
stacking_rmse = np.sqrt(mean_squared_error(train_y, stacking_oof_preds))

print(f"  Enhanced Stacking Meta-Model (LGBM) OOF RMSE: {stacking_rmse:.4f}")

# Memilih metode terbaik berdasarkan OOF RMSE
if weighted_ensemble_rmse < stacking_rmse:
    print("\nKeputusan: Memilih Optimal Weighted Ensemble karena memiliki OOF RMSE yang lebih rendah.")
    final_predictions = weighted_ensemble_test
else:
    print("\nKeputusan: Memilih Enhanced Stacking Meta-Model karena memiliki OOF RMSE yang lebih rendah.")
    final_predictions = meta_model_lgbm.predict(test_preds_ext)

# -----------------------------------------------------------------
# 7. Post-Processing & Submission
# -----------------------------------------------------------------
print("\n--- Final Prediction ---")

final_predictions = np.maximum(final_predictions, 0)

submission_df = pd.DataFrame({
    'date': submission_dates,
    'daily_ktCO2': final_predictions
})

submission_df.to_csv('submission_final_improved.csv', index=False)

print(f"Final predictions saved to submission_final_improved.csv")
print(f"Final prediction shape: {submission_df.shape}")


SIMPLE STACKING ENSEMBLE (Base + Meta)
--- Training Base Models & Generating OOF Predictions ---
Training lgbm...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001622 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6301
[LightGBM] [Info] Number of data points in the train set: 1284, number of used features: 59
[LightGBM] [Info] Start training from score 47.858568
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000504 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6265
[LightGBM] [Info] Number of data points in the train set: 1285, number of used features: 59
[LightGBM] [Info] Start training from score 47.871063
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000505 seconds.
You can set `force_row_wise=true`